**Navigation** : [Index](README.md) | [Audit de corpus (20) >>](Sudoku-20-CorpusAudit-Python.ipynb)

# Sudoku-19 — Soundness de la propagation de contraintes (companion formel natif)

Ce notebook est le **companion formel** du lake [`sudoku_lean`](sudoku_lean/),
dont la lib `Sudoku` prouve la **soundness des règles de propagation** d'un Sudoku
(*naked single*, *hidden single*) : ces règles, utilisées par tous les solveurs
(backtracking, OR-Tools, .NET), **ne retirent aucune solution valide** — elles
n'éliminent que des affectations impossibles. Résultat central de la série Sudoku
(issue #4055), avec **zéro `sorry`**.

L'idée : une cellule affectée **exclut sa valeur** de toutes ses cellules *paires*
(clé de voûte `peer_excludes_value`). De là dérivent par l'absurde les deux règles
de propagation, qui ne perdent donc jamais une solution.

## Convention de vérification — `#check` *natif* dans le kernel Lean

Ce notebook est un **notebook Lean natif** (kernel `lean4-wsl`) : il `import`e les
modules du lake directement et le compilateur Lean rend les signatures **dans le
notebook**. C'est rendu possible par l'UNLOCK (patch `lean4_jupyter` + jonction
Mathlib #2611).

> ⚠️ À l'exécution, la première cellule (`import`) peut prendre **~3 min** :
> le kernel charge les oleans Mathlib via la jonction NTFS. Les suivantes sont
> instantanées.

## 1. Import des modules du lake `sudoku_lean`

Le lake est structuré en 2 modules (`Basic`, `Propagation`) sous le namespace
`Sudoku`. On importe les 2 modules (la config `submodules` du lake ne build pas
l'umbrella racine — on cible directement les modules qui portent les définitions).

In [1]:
import Sudoku.Basic
import Sudoku.Propagation
import Sudoku.ExactCover
import Sudoku.Certificate
open Sudoku

import Sudoku.Basic
import Sudoku.Propagation
import Sudoku.ExactCover
import Sudoku.Certificate
open Sudoku
--% env 0

Raw input:
{"cmd": "import Sudoku.Basic\nimport Sudoku.Propagation\nimport Sudoku.ExactCover\nimport Sudoku.Certificate\nopen Sudoku"}
Raw output:
{"env": 0}

## 2. Preuve sans `sorry` — `#print axioms`

La clé de voûte `peer_excludes_value` et les deux théorèmes de soundness qui en dérivent
(`naked_single_sound`, `hidden_single_sound`) ne dépendent que de **deux axiomes**
(`propext`, `Quot.sound`) — pas de `sorryAx` et pas même de `Classical.choice` — ce qui
prouve que la preuve est **complète** (0 sorry) et **constructive** (par l'absurde via
`Classical.em`, dérivable de `propext` + `Quot.sound`). Le lemme `full_house_present`
(pigeonhole) utilise `Classical.choice` (`card_image_of_injOn`), donc 3 axiomes.

In [2]:
#print axioms peer_excludes_value
#print axioms naked_single_sound
#print axioms hidden_single_sound

#print axioms peer_excludes_value
──────▶  'Sudoku.peer_excludes_value' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms naked_single_sound
──────▶  'Sudoku.naked_single_sound' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms hidden_single_sound
──────▶  'Sudoku.hidden_single_sound' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "#print axioms peer_excludes_value\n#print axioms naked_single_sound\n#print axioms hidden_single_sound", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Sudoku.peer_excludes_value' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'Sudoku.naked_single_sound' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'Sudoku.hidden_single_sound' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 1}

## 3. La structure de contraintes abstraite d'un Sudoku

Le module `Sudoku/Basic.lean` formalise un Sudoku de façon **abstraite** : tout CSP
« à portées toutes-distinctes ». Le 9×9 concret (lignes, colonnes, blocs) en est une
instance, non un cas spécial — les théorèmes de soundness valent pour toute structure
de ce type.

- `Scopes ι = Finset (Finset ι)` — un ensemble fini de **portées** (chacune un `Finset` de cellules).
- `Solution ι V = ι → V` — une affectation complète (une valeur par cellule).
- `AllDistinctOn σ s` — `σ` est **toutes-distinctes sur `s`** (injectivité de `σ` sur `s`).
- `IsSolution scopes σ` — `σ` est toutes-distinctes sur **chacune** des portées (invariant fondamental).
- `IsPeer scopes c c'` — `c'` est une **paire** de `c` (distinctes, partagent une portée).
- `PresentIn σ s v` — la valeur `v` est présente dans la portée `s`.

Le lemme `full_house_present` (pigeonhole) : une portée « pleine maison »
(`s.card = card V`) porte toute valeur dans toute solution.

In [3]:
#check Scopes
#check Solution
#check AllDistinctOn
#check IsSolution
#check IsPeer
#check PresentIn
#check full_house_present

#check Scopes
──────▶  Sudoku.Scopes.{u_3} (ι : Type u_3) : Type u_3
#check Solution
──────▶  Sudoku.Solution.{u_3, u_4} (ι : Type u_3) (V : Type u_4) : Type (max u_3 u_4)
#check AllDistinctOn
──────▶  Sudoku.AllDistinctOn.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (σ : Solution ι V) (s : Finset ι) : Prop
#check IsSolution
──────▶  Sudoku.IsSolution.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (scopes : Scopes ι) (σ : Solution ι V) : Prop
#check IsPeer
──────▶  Sudoku.IsPeer.{u_1} {ι : Type u_1} (scopes : Scopes ι) (c c' : ι) : Prop
#check PresentIn
──────▶  Sudoku.PresentIn.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (σ : Solution ι V) (s : Finset ι) (v : V) : Prop
#check full_house_present
──────▶  Sudoku.full_house_present.{u_3, u_4} {ι : Type u_3} {V : Type u_4} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (σ : Solution ι V) (s : Finset ι) (v : V) (hcard : s.card = Fintype.card V)
  (hAD : AllDistinctOn σ s) : PresentIn σ s v
--% env 2

Raw input:
{"cmd": "#check Scopes\n#check Solution\n#check AllDistinctOn\n#check IsSolution\n#check IsPeer\n#check PresentIn\n#check full_house_present", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Sudoku.Scopes.{u_3} (ι : Type u_3) : Type u_3"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sudoku.Solution.{u_3, u_4} (ι : Type u_3) (V : Type u_4) : Type (max u_3 u_4)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sudoku.AllDistinctOn.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (σ : Solution ι V) (s : Finset ι) : Prop"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Sudoku.IsSolution.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (scopes : Scopes ι) (σ : Solution ι V) : Prop"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Sudoku.IsPeer.{u_1} {ι : Type u_1} (scopes : Scopes ι) (c c' : ι) : Prop"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "Sudoku.PresentIn.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (σ : Solution ι V) (s : Finset ι) (v : V) : Prop"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "Sudoku.full_house_present.{u_3, u_4} {ι : Type u_3} {V : Type u_4} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (σ : Solution ι V) (s : Finset ι) (v : V) (hcard : s.card = Fintype.card V)\n  (hAD : AllDistinctOn σ s) : PresentIn σ s v"}],
 "env": 2}

### Lecture : modéliser un CSP à portées toutes-distinctes

| Symbole Lean | Lecture |
|-------------|---------|
| `Scopes ι` | ensemble fini de portées (`Finset (Finset ι)`) |
| `Solution ι V` | affectation : une valeur par cellule (`ι → V`) |
| `AllDistinctOn σ s` | `σ` injective sur la portée `s` |
| `IsSolution scopes σ` | `σ` valide sur toutes les portées |
| `IsPeer scopes c c'` | `c'` paire de `c` (partagent une portée) |
| `full_house_present` | portée pleine maison ⟹ toute valeur présente (pigeonhole) |

## 4. La clé de voûte et la soundness des règles de propagation

Le module `Sudoku/Propagation.lean` prouve la **soundness** des deux règles canoniques
de propagation, à partir d'une unique clé de voûte :

- **`peer_excludes_value`** (clé de voûte) : une cellule `c` portant `v` **exclut `v`**
  de toute paire `c'` (`σ c' ≠ v`). Preuve : `c` et `c'` partagent une portée ; `σ`
  toutes-distinctes dessus ⟹ `σ c = σ c'` impliquerait `c = c'`, contredisant `c ≠ c'`.
- **`naked_single_sound`** : si toutes les valeurs sauf `v` sont déjà portées par des
  paires de `c`, alors `σ c = v` (par l'absurde via `peer_excludes_value`).
- **`hidden_single_sound`** : si `v` est présente dans la portée `s` et exclue de toute
  autre cellule de `s`, alors `σ c = v` où `c` est l'unique cellule possible.

Ces règles **ne retirent aucune solution valide** : elles identifient des valeurs
**forcées**.

In [4]:
#check peer_excludes_value
#check naked_single_sound
#check hidden_single_sound

#check peer_excludes_value
──────▶  Sudoku.peer_excludes_value.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (c c' : ι) (v : V)
  (hpeer : IsPeer scopes c c') (hcv : σ c = v) : σ c' ≠ v
#check naked_single_sound
──────▶  Sudoku.naked_single_sound.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (c : ι) (v : V)
  (hcover : ∀ (w : V), w ≠ v → ∃ c', IsPeer scopes c c' ∧ σ c' = w) : σ c = v
#check hidden_single_sound
──────▶  Sudoku.hidden_single_sound.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (s : Finset ι) (_hs : s ∈ scopes)
  (c : ι) (v : V) (_hcs : c ∈ s) (hvin : PresentIn σ s v)
  (hexcl : ∀ c' ∈ s, c' ≠ c → ∃ c'', IsPeer scopes c' c'' ∧ σ c'' = v) : σ c = v
--% env 3

Raw input:
{"cmd": "#check peer_excludes_value\n#check naked_single_sound\n#check hidden_single_sound", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sudoku.peer_excludes_value.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (c c' : ι) (v : V)\n  (hpeer : IsPeer scopes c c') (hcv : σ c = v) : σ c' ≠ v"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sudoku.naked_single_sound.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (c : ι) (v : V)\n  (hcover : ∀ (w : V), w ≠ v → ∃ c', IsPeer scopes c c' ∧ σ c' = w) : σ c = v"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sudoku.hidden_single_sound.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ) (s : Finset ι) (_hs : s ∈ scopes)\n  (c : ι) (v : V) (_hcs : c ∈ s) (hvin : PresentIn σ s v)\n  (hexcl : ∀ c' ∈ s, c' ≠ c → ∃ c'', IsPeer scopes c' c'' ∧ σ c'' = v) : σ c = v"}],
 "env": 3}

### Lecture : la propagation ne perd aucune solution

| Théorème | Conclusion |
|----------|-----------|
| `peer_excludes_value` | cellule affectée exclut sa valeur de ses paires (clé de voûte) |
| `naked_single_sound` | un seul candidat `v` restant ⟹ `σ c = v` |
| `hidden_single_sound` | `v` n'a qu'une cellule possible dans `s` ⟹ `σ c = v` |

### Le jalon restant (honnêtement)

La **complétude** des règles de propagation (suffisent-elles à résoudre tout Sudoku ?
non en général — d'où le backtracking en complément) reste un jalon naturel ouvert,
délibérément non `sorry`-backed : la lib reste entièrement `sorry`-free. La
**réduction à la couverture exacte** est désormais **prouvée** (section suivante,
`sudoku_iff_exact_cover`). Le résultat « 17 indices minimum » est hors scope (calcul
massif, non formalisable).

## 5. La chaîne causale complète

Les deux modules composent une chaîne unique, de la structure abstraite à la soundness :

1. **`Basic`** — `Scopes`/`Solution`/`AllDistinctOn`/`IsSolution`/`IsPeer`/`PresentIn`
   + `full_house_present` (pigeonhole : portée pleine maison ⟹ toute valeur présente).
2. **`Propagation`** — `peer_excludes_value` (clé de voûte) ⟶ `naked_single_sound` +
   `hidden_single_sound` (soundness des deux règles, par l'absurde).
3. **`ExactCover`** — `toSelection` + lemme-clé `mem_toSelection_iff` ⟶
   `solution_imp_exact_cover` (sens direct) + `exact_cover_imp_solution` (sens retour)
   ⟶ **`sudoku_iff_exact_cover`** (équivalence complète avec la couverture exacte de Knuth,
   sous hypothèse « pleine maison » satisfaite par le 9×9).

## 6. La réduction Sudoku ⇔ couverture exacte (Knuth)

La couverture exacte (Knuth 2000, « Dancing Links ») encode le Sudoku comme un
problème universel : chaque contrainte (cellule, portée×valeur) est un « élément »
à couvrir, chaque placement `(c, v)` une option. Une solution Sudoku est alors
exactement une **couverture exacte** — chaque élément couvert une et une seule fois.

Le théorème capstone `sudoku_iff_exact_cover` établit l'**équivalence complète**,
sous l'hypothèse « pleine maison » `∀ s ∈ scopes, s.card = card V` (satisfaite par
le 9×9 où chaque portée a 9 cellules pour 9 valeurs) :

`(∃ σ, IsSolution scopes σ) ↔ (∃ sel, IsExactCover sel scopes)`.

Lemme-clé `mem_toSelection_iff` : `(c, v) ∈ toSelection σ ↔ σ c = v` — toute question
sur la sélection se ramène à une question sur `σ`. Le sens direct utilise la pige
`full_house_present` (existence) + `AllDistinctOn` (unicité) ; le sens retour
construit l'affectation inverse `fromSelection` et montre que deux cellules d'une
même portée portant la même valeur violeraient l'unicité `∃!` de la paire (portée, valeur).

**0 sorry, axiomes = trio standard du noyau Lean** (`propext`, `Classical.choice`,
`Quot.sound`) — `Classical.choice` pour la construction non constructive
`fromSelection`, aucun axiome ad hoc.

In [5]:
#check ExactCover.IsExactCover
#check toSelection
#check mem_toSelection_iff
#check solution_imp_exact_cover
#check exact_cover_imp_solution
#check sudoku_iff_exact_cover
#print axioms sudoku_iff_exact_cover


#check ExactCover.IsExactCover
──────▶  Sudoku.ExactCover.IsExactCover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (sel : ExactCover.Sel ι V) (scopes : Scopes ι) :
  Prop
#check toSelection
──────▶  Sudoku.toSelection.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [DecidableEq V]
  (σ : Solution ι V) : ExactCover.Sel ι V
#check mem_toSelection_iff
──────▶  Sudoku.mem_toSelection_iff.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (σ : Solution ι V) (c : ι) (v : V) : (c, v) ∈ toSelection σ ↔ σ c = v
#check solution_imp_exact_cover
──────▶  Sudoku.solution_imp_exact_cover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ)
  (hfull : ∀ s ∈ scopes, s.card = Fintype.card V) : ExactCover.IsExactCover (toSelection σ) scopes
#check exact_cover_imp_solution
──────▶  Sudoku.exact_cover_imp_solution.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (sel : ExactCover.Sel ι V) (hec : ExactCover.IsExactCover sel scopes) :
  ∃ σ, IsSolution scopes σ ∧ toSelection σ = sel
#check sudoku_iff_exact_cover
──────▶  Sudoku.sudoku_iff_exact_cover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]
  [DecidableEq V] (scopes : Scopes ι) (hfull : ∀ s ∈ scopes, s.card = Fintype.card V) :
  (∃ σ, IsSolution scopes σ) ↔ ∃ sel, ExactCover.IsExactCover sel scopes
#print axioms sudoku_iff_exact_cover
──────▶  'Sudoku.sudoku_iff_exact_cover' depends on axioms: [propext, Classical.choice, Quot.sound]

--% env 4

Raw input:
{"cmd": "#check ExactCover.IsExactCover\n#check toSelection\n#check mem_toSelection_iff\n#check solution_imp_exact_cover\n#check exact_cover_imp_solution\n#check sudoku_iff_exact_cover\n#print axioms sudoku_iff_exact_cover\n", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sudoku.ExactCover.IsExactCover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} (sel : ExactCover.Sel ι V) (scopes : Scopes ι) :\n  Prop"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sudoku.toSelection.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [DecidableEq V]\n  (σ : Solution ι V) : ExactCover.Sel ι V"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sudoku.mem_toSelection_iff.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (σ : Solution ι V) (c : ι) (v : V) : (c, v) ∈ toSelection σ ↔ σ c = v"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Sudoku.solution_imp_exact_cover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (σ : Solution ι V) (hσ : IsSolution scopes σ)\n  (hfull : ∀ s ∈ scopes, s.card = Fintype.card V) : ExactCover.IsExactCover (toSelection σ) scopes"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Sudoku.exact_cover_imp_solution.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (sel : ExactCover.Sel ι V) (hec : ExactCover.IsExactCover sel scopes) :\n  ∃ σ, IsSolution scopes σ ∧ toSelection σ = sel"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "Sudoku.sudoku_iff_exact_cover.{u_1, u_2} {ι : Type u_1} {V : Type u_2} [Fintype ι] [DecidableEq ι] [Fintype V]\n  [DecidableEq V] (scopes : Scopes ι) (hfull : ∀ s ∈ scopes, s.card = Fintype.card V) :\n  (∃ σ, IsSolution scopes σ) ↔ ∃ sel, ExactCover.IsExactCover sel scopes"}

## 7. Du solveur au certificat : vérifier sans refaire

Jusqu'ici, la chaîne formelle dit *ce qu'est* une solution (`IsSolution`) et *ce que la propagation préserve* (`naked_single_sound`, `hidden_single_sound`). Il manque le maillon pratique : comment un solveur — n'importe lequel, écrit dans n'importe quel langage — **convainc** un lecteur qu'il a trouvé une solution, sans qu'on ait à le croire sur parole ni à refaire sa recherche ?

La réponse est un **certificat** : la liste des 81 placements retenus. Le module `Sudoku/Certificate.lean` porte le **checker** qui décide, dans le kernel Lean, si cette liste est un certificat valide. Le producteur du certificat (la tranche Python `dlx_certificates.py`, extraite du moteur de `Sudoku-02-DancingLinks-Python.ipynb`) et le vérificateur sont deux programmes distincts : c'est cette séparation qui fait la valeur du dispositif.

### Ce que le checker contrôle, étage par étage

| Étage | Question posée | Ce qu'il ferme |
|-------|----------------|----------------|
| `rawDuplicate` | la liste brute contient-elle deux fois la **même paire** ? | le piège de la conversion en `Finset`, qui effacerait le doublon |
| `cellConflict` | deux placements distincts portent-ils sur la **même cellule** ? | l'ambiguïté « deux valeurs pour une case » |
| `completeness` | **chaque** case porte-t-elle une valeur ? | le certificat partiel présenté comme complet |
| `exactCover` | pour chaque portée et chaque valeur, exactement **une** case ? | la violation d'une ligne, colonne ou bloc |
| `clues` | chaque indice du puzzle est-il **respecté** ? | la solution d'un *autre* puzzle |

L'ordre d'examen compte : le doublon brut est vérifié **avant** toute conversion en ensemble, précisément parce que la conversion détruirait l'information qu'on cherche à voir.

### Pourquoi le checker est un théorème, et pas seulement un programme

Le point délicat d'un vérificateur est le suivant : « le programme répond `ok` » et « la liste est une couverture exacte » sont deux énoncés différents. Le premier ne vaut rien si le second ne lui est pas relié.

`checkOk_isExactCover` établit ce lien — l'acceptation **implique** `ExactCover.IsExactCover` — puis `checkOk_givesSolution` **réutilise** `exact_cover_imp_solution`, déjà prouvé à la section 6, pour conclure `IsSolution`. La boucle est fermée : un certificat accepté *est* une solution, au sens exact de la définition qu'emploie la propagation.

Une contrainte de conception mérite d'être notée : le checker est **calculable** de bout en bout (`List.finRange`, énumérations de listes), donc évaluable par `decide` sur un certificat concret. C'est aussi pourquoi la géométrie du plateau est énumérée par `List.finRange` et non par `Finset.toList`, qui est `noncomputable` dans cette version de Mathlib — un détail qui décide du caractère exécutable de la preuve.

In [6]:
open Sudoku

/-- Valeur de la grille solution canonique 9x9 : `(3r + r/3 + c) mod 9`. -/
def solValue (i : Fin 81) : Fin 9 :=
  ⟨(i.val / 9 * 3 + i.val / 9 / 3 + i.val % 9) % 9, Nat.mod_lt _ (by decide)⟩

/-- Les 81 placements de la grille solution canonique. -/
def solPlacements : List (Fin 81 × Fin 9) :=
  (List.finRange 81).map fun c => (c, solValue c)

/-- Certificat accepte : la grille solution, sans indice de puzzle. -/
def certOK : Certificate.RawCertificate (Fin 81) (Fin 9) :=
  { version := 1, puzzle := "", clues := [], placements := solPlacements }

/-- Rendu textuel du verdict (l'inductif `CheckOutcome` ne derive pas `Repr`). -/
def renderVerdict : Certificate.CheckOutcome → String
  | .ok => "ok"
  | .rawDuplicate => "rawDuplicate"
  | .cellConflict => "cellConflict"
  | .completeness => "completeness"
  | .exactCover => "exactCover"
  | .clues => "clues"

def scopes : List (Finset (Fin 81)) := Certificate.sudokuScopeList

-- 1. Le certificat correct est accepte.
#eval renderVerdict (Certificate.checkCertificate certOK scopes)

-- 2. La meme paire apparait deux fois : refuse au premier etage.
#eval renderVerdict (Certificate.checkCertificate
        { certOK with placements := (⟨0, by decide⟩, ⟨0, by decide⟩) :: solPlacements } scopes)

-- 3. Deux valeurs sur la meme cellule : refuse avant la completude.
#eval renderVerdict (Certificate.checkCertificate
        { certOK with placements := (⟨0, by decide⟩, ⟨1, by decide⟩) :: solPlacements } scopes)

-- 4. Une case manque : le certificat est incomplet.
#eval renderVerdict (Certificate.checkCertificate
        { certOK with placements := solPlacements.tail } scopes)

#check Certificate.checkCertificate
#check Certificate.checkOk_isExactCover
#check Certificate.checkOk_givesSolution
#print axioms Certificate.checkOk_givesSolution

open Sudoku

/-- Valeur de la grille solution canonique 9x9 : `(3r + r/3 + c) mod 9`. -/
def solValue (i : Fin 81) : Fin 9 :=
  ⟨(i.val / 9 * 3 + i.val / 9 / 3 + i.val % 9) % 9, Nat.mod_lt _ (by decide)⟩

/-- Les 81 placements de la grille solution canonique. -/
def solPlacements : List (Fin 81 × Fin 9) :=
  (List.finRange 81).map fun c => (c, solValue c)

/-- Certificat accepte : la grille solution, sans indice de puzzle. -/
def certOK : Certificate.RawCertificate (Fin 81) (Fin 9) :=
  { version := 1, puzzle := "", clues := [], placements := solPlacements }

/-- Rendu textuel du verdict (l'inductif `CheckOutcome` ne derive pas `Repr`). -/
def renderVerdict : Certificate.CheckOutcome → String
  | .ok => "ok"
  | .rawDuplicate => "rawDuplicate"
  | .cellConflict => "cellConflict"
  | .completeness => "completeness"
  | .exactCover => "exactCover"
  | .clues => "clues"

def scopes : List (Finset (Fin 81)) := Certificate.sudokuScopeList

-- 1. Le certificat correct est accepte.
#eval renderVerdict (Certificate.checkCertificate certOK scopes)
─────▶  "ok"

-- 2. La meme paire apparait deux fois : refuse au premier etage.
#eval renderVerdict (Certificate.checkCertificate
─────▶  "rawDuplicate"
        { certOK with placements := (⟨0, by decide⟩, ⟨0, by decide⟩) :: solPlacements } scopes)

-- 3. Deux valeurs sur la meme cellule : refuse avant la completude.
#eval renderVerdict (Certificate.checkCertificate
─────▶  "cellConflict"
        { certOK with placements := (⟨0, by decide⟩, ⟨1, by decide⟩) :: solPlacements } scopes)

-- 4. Une case manque : le certificat est incomplet.
#eval renderVerdict (Certificate.checkCertificate
─────▶  "completeness"
        { certOK with placements := solPlacements.tail } scopes)

#check Certificate.checkCertificate
──────▶  Sudoku.Certificate.checkCertificate (cert : Certificate.RawCertificate (Fin 81) (Fin 9))
  (scopeList : List (Finset (Fin 81))) : Certificate.CheckOutcome
#check Certificate.checkOk_isExactCover
──────▶  Sudoku.Certificate.checkOk_isExactCover (cert : Certificate.RawCertificate (Fin 81) (Fin 9))
  (scopeList : List (Finset (Fin 81)))
  (hok : Certificate.checkCertificate cert scopeList = Certificate.CheckOutcome.ok) :
  ExactCover.IsExactCover cert.placements.toFinset scopeList.toFinset
#check Certificate.checkOk_givesSolution
──────▶  Sudoku.Certificate.checkOk_givesSolution (cert : Certificate.RawCertificate (Fin 81) (Fin 9))
  (hok : Certificate.checkCertificate cert Certificate.sudokuScopeList = Certificate.CheckOutcome.ok) :
  ∃ σ, IsSolution Certificate.sudokuScopes σ ∧ toSelection σ = cert.placements.toFinset
#print axioms Certificate.checkOk_givesSolution
──────▶  'Sudoku.Certificate.checkOk_givesSolution' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 5

Raw input:
{"cmd": "open Sudoku\n\n/-- Valeur de la grille solution canonique 9x9 : `(3r + r/3 + c) mod 9`. -/\ndef solValue (i : Fin 81) : Fin 9 :=\n  \u27e8(i.val / 9 * 3 + i.val / 9 / 3 + i.val % 9) % 9, Nat.mod_lt _ (by decide)\u27e9\n\n/-- Les 81 placements de la grille solution canonique. -/\ndef solPlacements : List (Fin 81 \u00d7 Fin 9) :=\n  (List.finRange 81).map fun c => (c, solValue c)\n\n/-- Certificat accepte : la grille solution, sans indice de puzzle. -/\ndef certOK : Certificate.RawCertificate (Fin 81) (Fin 9) :=\n  { version := 1, puzzle := \"\", clues := [], placements := solPlacements }\n\n/-- Rendu textuel du verdict (l'inductif `CheckOutcome` ne derive pas `Repr`). -/\ndef renderVerdict : Certificate.CheckOutcome \u2192 String\n  | .ok => \"ok\"\n  | .rawDuplicate => \"rawDuplicate\"\n  | .cellConflict => \"cellConflict\"\n  | .completeness => \"completeness\"\n  | .exactCover => \"exactCover\"\n  | .clues => \"clues\"\n\ndef scopes : List (Finset (Fin 81)) := Certificate.sudokuScopeList\n\n-- 1. Le certificat correct est accepte.\n#eval renderVerdict (Certificate.checkCertificate certOK scopes)\n\n-- 2. La meme paire apparait deux fois : refuse au premier etage.\n#eval renderVerd

### Lecture : quatre certificats, quatre verdicts

La cellule précédente construit quatre certificats à partir de la **grille solution canonique** (`valeur(r, c) = (3r + r/3 + c) mod 9`) et demande son verdict au checker :

| Certificat | Construction | Verdict attendu |
|------------|--------------|-----------------|
| accepté | la grille solution, complète | `ok` |
| doublon brut | la paire `(0, 0)` ajoutée une seconde fois | `rawDuplicate` |
| conflit de cellule | `(0, 1)` ajouté devant `(0, 0)` | `cellConflict` |
| incomplet | la première case retirée de la liste | `completeness` |

Ce que la démonstration établit est **exactement** ce qu'un vérificateur doit garantir : la première liste est acceptée, et **chacune** des trois suivantes est rejetée à l'étage qui la concerne — pas rejetée « quelque part ». Un rejet générique ne dirait pas au producteur du certificat ce qu'il doit corriger.

Le `#print axioms` complète la démonstration sur un autre plan : il montre sur quels axiomes repose la chaîne de soundness. On y lit les axiomes du noyau et, le cas échéant, le principe du tiers exclu classique — mais jamais `sorryAx` (une preuve manquante) ni `native_decide` (une décision déléguée au compilateur sans preuve), qui videraient le théorème de son contenu.

## 8. Exemple guidé et exercices

### Exercice 1 — Paires et exclusion de candidats (Python)

Ancrez l'intuition : modélisez un mini-Sudoku comme un ensemble de portées, et
implémentez `is_peer` (deux cellules sont paires si distinctes ET partagent une portée)
puis `eliminate` (une cellule affectée exclut sa valeur de tous ses pairs).

```python
def is_peer(scopes, c, cprime):
    # c et cprime sont paires : distincts ET partagent au moins une portee
    if c == cprime:
        return False
    # TODO etudiant : existe-t-il une portee s contenant a la fois c et cprime ?
    return None  # TODO
```

### Exercice 2 — Une cellule n'est pas sa propre paire (Lean)

Prouvez qu'une cellule n'est jamais paire d'elle-même : `¬ IsPeer scopes c c`.
Indice : `IsPeer` exige `c ≠ c'` (première conjonction). Si `c = c'`, cette conjonction
est fausse.

```lean
-- TODO etudiant : remplacer le sorry
-- example : ¬ IsPeer scopes c c := by
--   intro h
--   exact h.1 rfl
```

### Exercice 3 — Naked single et hidden single (Python)

Mettez en évidence la différence entre les deux règles sur un mini-Sudoku (candidats =
ensemble de valeurs possibles par cellule). Implémentez `naked_single` (une cellule
n'a plus qu'un candidat) et `hidden_single` (une valeur n'a qu'une seule cellule
possible dans une portée). Vérifiez que les deux réduisent les candidats sans perte.

```python
def naked_single(candidates):
    # Naked single : une cellule dont le seul candidat restant est v doit valoir v
    # TODO etudiant : retourne {cell: v} pour les cellules a candidat unique
    return None  # TODO

def hidden_single(candidates, scopes):
    # Hidden single : une valeur n'allant que dans une cellule d'une portee y va
    # TODO etudiant : pour chaque portee et valeur, si une seule cellule peut la porter
    return None  # TODO
```

## Conclusion

Ce companion **natif** exhibe la preuve formelle 0-sorry de la soundness de la
propagation de contraintes Sudoku dans le kernel Lean lui-même : `#check` et
`#print axioms` rendent les signatures et les axiomes réels produits par le
compilateur, sans intermédiaire Python.

La clé de voûte `peer_excludes_value` fonde les deux règles `naked_single_sound` et
`hidden_single_sound` : la propagation **ne retire aucune solution valide**, elle
n'élimine que des affectations qu'aucune solution n'utilise. C'est le fondement formel
des solveurs Sudoku (backtracking, OR-Tools, .NET) enseignés dans la série.

**Jalon ouvert** : seule la complétude des règles (suffisent-elles à résoudre tout
Sudoku ?) reste non formalisée ; la lib reste sorry-free. La réduction à la couverture
exacte est **prouvée** (`sudoku_iff_exact_cover`, section 6).